# [수학 Retreat #8] 확률의 역동성: 대수의 법칙과 베이시안 필터

이 주피터 노트북은 중학교 2학년 과정의 **확률** 단원을 파이썬 2D 그래픽 시각화를 통해 시각적으로 탐구하는 실습 교재입니다.
한 치 앞을 모르는 불확실성 속에서 수학적 이성이 어떻게 데이터와 반복을 통해 질서를 찾아 나가는지 학습합니다.

### 🟢 본 실습에서 다룰 두 가지 주제:
1. **대수의 법칙 (`law_of_large_numbers`)**: 동전을 무수히 던질 때, 시행 횟수(N)가 적을 때는 확률이 요동치다가 N이 수만 번으로 확장됨에 따라 이론적 수학 확률인 $0.5$로 극적으로 수렴해 가는 수렴의 법칙 체험.
2. **베이즈 정리 기반 스팸 필터 (`bayes_spam_filter`)**: 메일 속에 포함된 키워드 데이터에 따라 가상 메일이 스팸(Spam)일 조건부 확률을 실시간으로 갱신(Update)하는 예측 대시보드 시뮬레이션.

**⚠️ 안내:** 본 파일은 파이썬 초보자도 코드 한 줄 한 줄의 기하학적·수학적 연산 원리를 이해할 수 있도록 **세포 단위의 멀티셀**로 쪼개어 구성했으며 상세한 한글 주석을 부착했습니다.


## 1. 환경 준비 및 필수 라이브러리 설치
본 실습에서는 확률 분포 난수 생성 및 수열 계산을 위해 `numpy`, 동적 실시간 플로팅을 위해 `plotly`, 그리고 대시보드 컴포넌트 제어를 위해 `ipywidgets`를 활용합니다.
아래 셀을 실행하여 라이브러리를 먼저 준비합니다.


In [ ]:
# !pip install 명령어는 주피터 노트북 내에서 터미널의 패키지 매니저를 호출하여 패키지를 설치하게 하는 매직 명령어입니다.
# -q (quiet) 옵션을 붙여 백그라운드에서 조용히 설치를 끝마칩니다.
%pip install -q numpy plotly ipywidgets


### 1.2 모듈 로드 (Import)
확률 시뮬레이션 및 데이터 시각화, 제어반 구동에 필요한 모듈들을 메모리에 로드합니다.


In [ ]:
# numpy: 동전 던지기 난수 시뮬레이션 및 확률 누적 합산 등을 계산할 파이썬의 표준 수치 라이브러리입니다.
import numpy as np

# plotly.graph_objects: 동적 그래프 플로팅과 아름다운 확률 수렴 라인 차트 표현을 지원하는 그래프 빌더 모듈입니다.
import plotly.graph_objects as go

# ipywidgets: 사용자가 동전 던지는 횟수를 가변 조절하거나 단어 포함 여부를 체크할 수 있도록 제어 위젯을 제공합니다.
import ipywidgets as widgets

# IPython.display: 스팸 확률 계산 대시보드 리포트를 위해 스타일링된 예쁜 HTML을 주피터 내에 출력해 줍니다.
from IPython.display import display, HTML


---
## 2. 실습 1: 대수의 법칙 시뮬레이터 (`law_of_large_numbers`)
**대수의 법칙(Law of Large Numbers)**은 어떤 사건을 무한히 많이 반복하여 시행할 때,
실제 측정된 **상대도수(통계적 확률)**가 시행 횟수가 커질수록 **이론적 확률(수학적 확률)**에 완벽하게 수렴해 간다는 정리입니다.

- **시행 횟수(N)가 적을 때 (10 ~ 50회)**: 동전의 앞면이 나올 확률은 $0.2$나 $0.9$처럼 일시적인 노이즈로 인해 크게 요동칩니다.
- **시행 횟수(N)가 많아질 때 (10,000회 이상)**: 임의성이 제거되고 우주의 거대한 확률 질서가 드러나며 수학적 값인 정확히 **0.5 (50%)**에 자석처럼 부드럽게 수렴하게 됩니다.

동전의 앞면(1), 뒷면(0) 난수를 무작위로 던져 각 시행 시점마다의 누적 앞면 비율을 계산하는 시뮬레이션 연산 함수를 정의합니다.


In [ ]:
def simulate_coin_tosses(max_trials):
    """
    지정된 최대 시행 횟수(max_trials)만큼 동전 던지기 난수 시뮬레이션을 돌려
    각 시행 시점별 누적 앞면 비율(통계적 확률) 수열을 계산하여 리턴합니다.
    """
    # 난수 생성의 일치성을 보장하기 위해 랜덤 시드를 42로 고정합니다.
    np.random.seed(42)
    
    # 1. 0(뒷면) 또는 1(앞면)을 갖는 동전 던지기 난수 결과를 max_trials 개수만큼 생성합니다.
    # np.random.randint(low, high, size): low 이상 high 미만의 정수 생성
    toss_results = np.random.randint(0, 2, size=max_trials)
    
    # 2. np.cumsum 함수를 사용해 동전 앞면(1)의 누적 횟수를 순서대로 더해 나갑니다.
    # 예: [1, 0, 1, 1] -> 누적 합 [1, 1, 2, 3]
    cumulative_heads = np.cumsum(toss_results)
    
    # 3. 각 시행 번호(1부터 max_trials까지) 배열을 생성하여 누적 합을 나누어 줌으로써 누적 상대도수를 구합니다.
    trial_numbers = np.arange(1, max_trials + 1)
    cumulative_ratios = cumulative_heads / trial_numbers
    
    return trial_numbers, cumulative_ratios


### 2.2 실시간 상대도수 및 수렴도 리포트 대시보드
시행 횟수에 맞춰 최종 통계적 확률이 이론적 확률(50%)과 얼마나 오차가 줄어들었는지를 요약하고,
대수의 법칙이 완성되는 과정을 단계별로 시각 정보로 변환해 주는 HTML 대시보드 함수를 정의합니다.


In [ ]:
def build_lln_stats_html(trials, final_ratio):
    """
    현재 시뮬레이션의 최종 오차 및 진행도를 시각화한 대시보드를 생성합니다.
    """
    theoretical_prob = 0.5
    error = abs(final_ratio - theoretical_prob)
    
    # 1. 시행 범위에 따른 진단 메시지 맵핑
    if trials < 100:
        state_color = "#E11D48"  # 빨강 (불안정)
        state_msg = "❌ <b>초기 상태 (극심한 불확실성):</b> 시행 횟수가 너무 적어 일시적인 운과 노이즈에 의해 확률이 요동칩니다."
    elif trials < 1000:
        state_color = "#F59E0B"  # 주황 (과도기)
        state_msg = "⚠️ <b>중기 상태 (과도기적 수렴):</b> 데이터가 서서히 쌓이면서 진폭이 점점 줄어들고 있습니다."
    else:
        state_color = "#2FA85D"  # 초록 (수렴 완성)
        state_msg = "✅ <b>대수 상태 (필연적 수렴):</b> 시행 횟수(N)가 충분히 커지자 임의성이 제거되며 우주의 확률 법칙인 50%에 수렴합니다."
        
    # 2. HTML 템플릿 조립
    html_template = f"""
    <div style='background: rgba(255, 255, 255, 0.85);
                backdrop-filter: blur(8px);
                border: 1.5px solid rgba(11, 121, 208, 0.25);
                border-radius: 14px; padding: 20px; max-width: 650px;
                font-family: sans-serif; box-shadow: 0 4px 15px rgba(0,0,0,0.06);'>
        <h4 style='margin: 0 0 10px 0; color: #0B79D0;'>📊 대수의 법칙 실시간 수렴도 분석</h4>
        <div style='display: flex; justify-content: space-around; text-align: center; margin-bottom: 12px;'>
            <div>
                <span style='color: #666; font-size: 0.85em;'>총 시행 횟수 (N)</span>
                <h3 style='margin: 5px 0 0 0; color: #1F2937;'>{trials:,} 회</h3>
            </div>
            <div style='border-left: 1px solid #E2E8F0; padding-left: 20px;'>
                <span style='color: #666; font-size: 0.85em;'>측정된 상대도수</span>
                <h3 style='margin: 5px 0 0 0; color: {state_color};'>{final_ratio * 100:.2f} %</h3>
            </div>
            <div style='border-left: 1px solid #E2E8F0; padding-left: 20px;'>
                <span style='color: #666; font-size: 0.85em;'>이론과의 절대오차</span>
                <h3 style='margin: 5px 0 0 0; color: #475569;'>{error:.5f}</h3>
            </div>
        </div>
        <div style='background: rgba(0,0,0,0.02); padding: 10px; border-radius: 8px; font-size: 0.9em; border-left: 4px solid {state_color};'>
            {state_msg}
        </div>
    </div>
    """
    return html_template


### 2.3 Plotly 2D 대수의 법칙 수렴 그래프 구현
x축을 시행 횟수, y축을 누적 앞면 비율로 설정한 파란색 꺾은선 차트를 그립니다.
이때, 정확한 수학적 기준선인 0.5 ($50\%$) 높이의 기준선을 빨간색 점선으로 함께 겹쳐 그려 수렴해 가는 모습을 시각적으로 부각합니다.


In [ ]:
def render_large_numbers_plot(trials_count):
    """
    사용자가 슬라이더로 조절한 시행 횟수만큼 난수를 구동하여 누적 상대도수 수렴 그래프를 그립니다.
    """
    # 1. 동전 던지기 수열 데이터 획득
    numbers, ratios = simulate_coin_tosses(trials_count)
    final_rate = ratios[-1]
    
    # 2. HTML 통계 대시보드 리포팅 표출
    display(HTML(build_lln_stats_html(trials_count, final_rate)))
    
    # 3. 실시간 통계적 확률 변화 꺾은선 트레이스 (브랜드 블루 컬러)
    trace_simulation = go.Scatter(
        x=numbers,
        y=ratios,
        mode='lines',
        line=dict(color='#0B79D0', width=2),
        name='측정된 누적 상대도수 (통계적 확률)'
    )
    
    # 4. 고정 수학적 이론 확률선 (50% 기준선, 빨간색 점선)
    trace_theoretical_line = go.Scatter(
        x=[1, trials_count],
        y=[0.5, 0.5],
        mode='lines',
        line=dict(color='#E11D48', width=2, dash='dash'),
        name='이론적 확률 기준선 (0.50)'
    )
    
    # 레이아웃 정의
    layout = go.Layout(
        title=dict(text='<b>대수의 법칙 동전 던지기 확률 수렴 시뮬레이터</b>', x=0.5),
        xaxis=dict(title='동전 던진 횟수 (N)', gridcolor='#F1F5F9', type='linear'),
        yaxis=dict(title='누적 앞면 비율', range=[0.2, 0.8], gridcolor='#F1F5F9', dtick=0.1),
        plot_bgcolor='white',
        width=750, height=450,
        margin=dict(l=50, r=30, b=40, t=65),
        showlegend=True,
        legend=dict(orientation='h', y=-0.2, x=0.5, xanchor='center')
    )
    
    fig = go.Figure(data=[trace_simulation, trace_theoretical_line], layout=layout)
    fig.show()


### 2.4 대수의 법칙 인터랙티브 슬라이더 작동
시행 횟수를 10회부터 최대 15,000회까지 조절하는 정수 슬라이더 위젯을 조립하여 시뮬레이션을 작동시킵니다.
슬라이더를 우측으로 넓혀가며 거칠게 요동치던 파란 수열선이 수학적 질서(빨간 점선)에 어떻게 자석처럼 결합하는지 통찰해 보세요.


In [ ]:
# 시행 횟수를 10회부터 15,000회까지 조작 가능한 로그 스케일 지향적 슬라이더 위젯 생성
trials_slider = widgets.IntSlider(
    value=50, # 최초 실행 시 50회 설정 (요동치는 상태)
    min=10, max=15000, step=50,
    description='시행 횟수 (N):',
    style={'description_width': 'initial'},
    continuous_update=False # 슬라이더를 놓고 정지했을 때 연산 및 렌더링 부하 최소화
)

# 위젯과 시뮬레이션 매핑 기동
widgets.interact(render_large_numbers_plot, trials_count=trials_slider);


---
## 3. 실습 2: 베이즈 정리 기반 스팸 메일 예측 대시보드 (`bayes_spam_filter`)
**베이즈 정리(Bayes' Theorem)**는 새로운 정보나 데이터가 유입될 때마다, 기존 사건에 대한 믿음(확률)을 업데이트하는 **조건부 확률** 공식입니다.

### 수학적 공식:
$$P(\text{Spam} | \text{Word}) = \frac{P(\text{Word} | \text{Spam}) P(\text{Spam})}{P(\text{Word})}$$

여러 개의 키워드 $W_1, W_2, \dots$ 가 동시에 메일에서 발견되었을 때, 이 메일이 스팸일 사후 확률(Posterior Probability)은 나이브 베이즈(Naive Bayes) 이론을 적용해 다음과 같이 계산합니다:
$$P(\text{Spam} | W_1, W_2, \dots) = \frac{P(\text{Spam}) \prod P(W_i | \text{Spam})}{P(\text{Spam}) \prod P(W_i | \text{Spam}) + P(\text{Ham}) \prod P(W_i | \text{Ham})}$$

이 공식에 사용할 사전 수치 데이터(Prior Probabilities)를 다음과 같이 정의합니다:
- **$P(\text{Spam})$ (전체 메일 중 스팸 비율)**: $0.4$ (40%)
- **$P(\text{Ham})$ (전체 메일 중 정상 메일 비율)**: $0.6$ (60%)
- **스팸/정상 메일 내부에서의 개별 단어 출현 확률**:
  - '광고' : 스팸일 때 나타날 확률 $0.8$ | 정상일 때 $0.1$
  - '무료' : 스팸일 때 나타날 확률 $0.7$ | 정상일 때 $0.15$
  - '투자' : 스팸일 때 나타날 확률 $0.65$ | 정상일 때 $0.05$
  - '선물' : 스팸일 때 나타날 확률 $0.5$ | 정상일 때 $0.1$

메일에 포함될 단어들의 체크박스 유무 조건 신호를 받아 사후 확률을 연산하는 스팸 판정 엔진 함수를 정의합니다.


In [ ]:
# 스팸 필터링을 위한 사전 수치 매핑
P_Spam = 0.4 # 사전 확률 P(Spam)
P_Ham = 0.6  # 사전 확률 P(Ham)

# 단어별 조건부 확률 P(Word | Category) 매핑 사전
word_likelihoods = {
    '광고': {'Spam': 0.80, 'Ham': 0.10},
    '무료': {'Spam': 0.70, 'Ham': 0.15},
    '투자': {'Spam': 0.65, 'Ham': 0.05},
    '선물': {'Spam': 0.50, 'Ham': 0.10}
}

def calculate_spam_posterior(active_keywords):
    """
    체크된 단어 목록(active_keywords)을 기반으로 메일이 스팸일 사후 확률을 계산합니다.
    """
    # 만약 아무 단어도 체크되지 않았다면 사전 확률인 0.4(40%)를 그대로 반환합니다.
    if len(active_keywords) == 0:
        return P_Spam
        
    # 나이브 베이즈 확률 계산을 위한 우도 누적 곱 초기값 설정
    likelihood_spam_prod = 1.0
    likelihood_ham_prod = 1.0
    
    # 4가지 키워드 전체 리스트에 대해 포함 여부 추적
    # 체크된 단어는 해당 단어가 나타날 확률 P(W|S), 체크되지 않은 단어는 나타나지 않을 확률인 여사건 확률 1 - P(W|S)를 곱해 줍니다.
    for word, probabilities in word_likelihoods.items():
        if word in active_keywords:
            # 단어가 나타남
            likelihood_spam_prod *= probabilities['Spam']
            likelihood_ham_prod *= probabilities['Ham']
        else:
            # 단어가 나타나지 않음 (여사건 적용)
            likelihood_spam_prod *= (1.0 - probabilities['Spam'])
            likelihood_ham_prod *= (1.0 - probabilities['Ham'])
            
    # 분자: P(Spam) * P(W1|Spam) * P(W2|Spam) * ...
    numerator = P_Spam * likelihood_spam_prod
    
    # 분모: P(Spam) * P(W|Spam) + P(Ham) * P(W|Ham)
    denominator = numerator + (P_Ham * likelihood_ham_prod)
    
    # 사후 확률 연산
    posterior_prob = numerator / denominator
    
    return posterior_prob


### 3.2 게이지바 및 실시간 베이지안 사후 확률 대시보드 설계
데이터 유입에 따른 스팸 확률 수치의 실시간 갱신 현상을 직관적으로 나타내기 위해,
Plotly Horizontal Bar를 활용한 가로형 **스팸 게이지 바**와 스타일링된 스팸 진단 리포트를 출력하는 함수를 구현합니다.
스팸 위험도가 50% 이상일 때는 경고 빨간색(`rgb(225, 29, 72)`), 미만일 때는 안전 녹색(`rgb(47, 168, 93)`)으로 게이지바의 컬러가 동적으로 변하게 연출합니다.


In [ ]:
def render_spam_filter_dashboard(has_ad, has_free, has_invest, has_gift):
    """
    체크박스들의 선택 여부에 따라 실시간 사후 확률 게이지바와 진단서를 렌더링합니다.
    """
    # 1. 활성화된 키워드 리스트 수집
    active_list = []
    if has_ad: active_list.append('광고')
    if has_free: active_list.append('무료')
    if has_invest: active_list.append('투자')
    if has_gift: active_list.append('선물')
    
    # 2. 베이지안 사후 확률 연산
    posterior_prob = calculate_spam_posterior(active_list)
    prob_percent = posterior_prob * 100
    
    # 3. 확률 크기에 따른 색상 및 판정 등급 정의
    if prob_percent >= 80:
        color = "#E11D48"  # 강렬한 레드 (스팸 확실)
        status_text = "🚨 스팸 확정 (SPAM) - 차단 권장"
    elif prob_percent >= 50:
        color = "#F59E0B"  # 주황 (의심)
        status_text = "⚠️ 스팸 의심 (SUSPICIOUS) - 검토 요망"
    else:
        color = "#2FA85D"  # 브랜드 그린 (정상)
        status_text = "✅ 정상 메일 (HAM) - 안전함"
        
    # 4. HTML 요약 진단 대시보드 빌드
    keywords_str = ', '.join([f"'<b>{w}</b>'" for w in active_list]) if active_list else "없음"
    dashboard_html = f"""
    <div style='background: white; border: 1.5px solid {color};
                padding: 18px; border-radius: 14px; max-width: 600px; margin-bottom: 12px;
                font-family: sans-serif; box-shadow: 0 4px 15px rgba(0,0,0,0.06);'>
        <h4 style='margin: 0 0 10px 0; color: #475569;'>📬 실시간 메일 스팸 감별 리포트</h4>
        <div style='font-size: 0.9em; color: #666; margin-bottom: 8px;'>
            검출된 스팸 키워드 데이터: <span style='color: {color}; font-weight: bold;'>{keywords_str}</span>
        </div>
        <div style='display: flex; align-items: center; justify-content: space-between;'>
            <span style='font-size: 1.2em; font-weight: bold; color: {color};'>{status_text}</span>
            <span style='font-size: 1.6em; font-weight: bold; color: #1F2937;'>{prob_percent:.2f} %</span>
        </div>
    </div>
    """
    display(HTML(dashboard_html))
    
    # 5. Plotly 수평 게이지바 구성
    gauge_trace = go.Bar(
        x=[prob_percent],
        y=['스팸 확률'],
        orientation='h', # 가로형 바 그래프 설정
        marker=dict(color=color),
        width=0.4,
        hovertemplate='<b>스팸 지수:</b> %{x:.2f}%<extra></extra>'
    )
    
    # 레이아웃 정의
    layout = go.Layout(
        xaxis=dict(title='확률 (%)', range=[0, 100], gridcolor='#E2E8F0'),
        yaxis=dict(showgrid=False),
        plot_bgcolor='white',
        width=650, height=200,
        margin=dict(l=60, r=30, b=40, t=20),
        showlegend=False
    )
    
    fig = go.Figure(data=[gauge_trace], layout=layout)
    fig.show()


### 3.3 조건부 확률 스팸 필터 위젯 조립 및 기동
메일 본문에 네 가지 단어가 포함되었는지 선택하는 4개의 체크박스 위젯을 배치하여 시뮬레이터를 가동합니다.
체크박스를 하나씩 클릭하여 **데이터(단어)가 추가될 때마다 사전 스팸 지수 40%가 어떻게 2.7%, 85%, 98.7%로 급격히 변해 들어가는지** 조건부 확률의 정보 업데이트 성질을 확인해 보세요.


In [ ]:
# 4가지 핵심 키워드 유무 체크박스 위젯 생성
chk_ad = widgets.Checkbox(value=False, description='"광고" 포함 여부')
chk_free = widgets.Checkbox(value=False, description='"무료" 포함 여부')
chk_invest = widgets.Checkbox(value=False, description='"투자" 포함 여부')
chk_gift = widgets.Checkbox(value=False, description='"선물" 포함 여부')

# 체크박스들을 가로 및 세로 배치로 구조 조립
widgets_control_panel = widgets.VBox([
    widgets.HTML("<h4 style='color:#0B79D0; margin:0 0 5px 0;'>✉️ 메일 내용 키워드 선택</h4>"),
    chk_ad, chk_free, chk_invest, chk_gift
])

# 체크박스 변수값들과 감별 대시보드 렌더러 연동
output_spam = widgets.interactive_output(
    render_spam_filter_dashboard,
    {
        'has_ad': chk_ad,
        'has_free': chk_free,
        'has_invest': chk_invest,
        'has_gift': chk_gift
    }
)

# 가로 레이아웃 배치 기동
display(widgets.HBox([widgets_control_panel, output_spam]))


---
## 4. 깊이 있는 기하학 사색을 위한 열린 질문 (Joshua를 위한 질문)

1. **대수의 법칙과 비즈니스 뚝심**:
   초기에 시행 횟수가 10~50회에 불과할 때는 우연과 노이즈로 인해 확률이 요동쳤지만, 무수히 반복된 15,000번의 누적 스케일 속에서 동전 앞면 확률은 결국 50%의 이론적 필연(Order)으로 수렴했습니다.
   단기적인 실패와 요동침(노이즈)에 일희일비하지 않고, 장기적인 비즈니스 공식과 우상향의 가치(대수의 법칙)를 구현하기 위해 Joshua님이 굳건히 지켜내야 할 '반복해야 할 올곧은 행위의 본질'은 무엇인지 사색해 보세요.

2. **베이즈 업데이트와 의사결정의 유연성**:
   최초에는 스팸일 확률이 40%에 불과했으나, '광고', '투자' 등의 추가 단어 데이터가 입력될 때마다 확률 지수는 85%, 98.7%로 실시간으로 갱신(Update)되어 최종 판단에 이르렀습니다.
   과거의 편견이나 초기 가설(사전 확률)에 고착되지 않고, 끊임없이 유입되는 새로운 팩트와 고객 피드백(우도)을 수용하여 내 비즈니스 방향성과 가설을 동적으로 보정해 나가는 '베이지안 의사결정'에 대해 느낀 철학적 소회를 나눠 주세요.
